<a href="https://colab.research.google.com/github/msfasha/307401-Big-Data/blob/main/20252/Module%205%20-%20Introduction%20to%20Spark/5-Spark_Structured_Streaming.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

<h1 style="color:#E25822; font-size:2.5em; font-weight:bold; margin-bottom:0.2em">
  Spark Structured Streaming
</h1>
<h3 style="color:#555; margin-top:0">Module 5 — Big Data Engineering</h3>

---

**Learning Objectives**

By the end of this notebook you will be able to:

- Explain the difference between batch and streaming data processing
- Describe the Spark Structured Streaming programming model
- Build streaming pipelines using the rate source and file-based sources
- Apply window operations and watermarking to handle time-based aggregations and late data
- Design a real-world streaming ETL pattern

> **Note on Running Streaming Queries in Notebooks**  
> True streaming queries run indefinitely. In this notebook every streaming query is wrapped with a short `time.sleep()` followed by `query.stop()` so that cells complete and you can run the notebook top to bottom. Each section explains what the output represents in a production setting.

<h2 style="color:#E25822">1. Streaming vs Batch Processing</h2>

Data arrives in two fundamentally different shapes:

| Dimension | **Batch Processing** | **Stream Processing** |
|---|---|---|
| Data scope | Bounded dataset (finite) | Unbounded dataset (infinite) |
| When it runs | Scheduled (hourly, nightly…) | Continuously, as events arrive |
| Latency | Minutes → hours | Milliseconds → seconds |
| Complexity | Lower | Higher (time, ordering, failures) |
| Typical tools | Spark batch, Hive, SQL | Spark Streaming, Flink, Kafka Streams |
| Example output | Daily sales report | Live fraud alert |

### Real-World Use Cases

| Domain | Streaming Use Case |
|---|---|
| **IoT / Industrial** | Sensor telemetry from thousands of machines; alert when temperature exceeds threshold |
| **E-Commerce** | Clickstream analysis — detect abandoned carts in real time and trigger a push notification |
| **Finance** | Fraud detection — flag transactions that deviate from a user's normal spending pattern |
| **Social Media** | Count trending hashtags in a 5-minute sliding window |
| **Logistics** | GPS pings from delivery vehicles; re-route when traffic delay > 10 min |
| **Healthcare** | Continuous ECG monitoring; alert clinical staff to arrhythmia within seconds |

### Architecture: From Source to Sink

```
┌──────────────────────────────────────────────────────────────────────┐
│                     STREAMING PIPELINE OVERVIEW                      │
└──────────────────────────────────────────────────────────────────────┘

  ┌────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
  │  SOURCES   │────▶│   INGEST /   │────▶│   PROCESS    │────▶│   SINKS      │
  │            │     │   MESSAGE    │     │   (Spark     │     │              │
  │ • Sensors  │     │   BROKER     │     │  Structured  │     │ • Dashboard  │
  │ • Logs     │     │ • Kafka      │     │  Streaming)  │     │ • Database   │
  │ • APIs     │     │ • Kinesis    │     │              │     │ • Data Lake  │
  │ • Files    │     │ • Event Hub  │     │ • Filter     │     │ • Kafka      │
  │ • Clicks   │     └──────────────┘     │ • Aggregate  │     │ • Alerts     │
  └────────────┘                          │ • Join       │     └──────────────┘
                                          │ • ML Infer   │
                                          └──────────────┘
                                                 │
                                          ┌──────────────┐
                                          │  CHECKPOINT  │
                                          │  (fault      │
                                          │  tolerance)  │
                                          └──────────────┘
```

<h2 style="color:#E25822">2. Spark Streaming Evolution</h2>

Spark has had **two generations** of streaming APIs:

### Generation 1 — DStreams (Spark Streaming, ~2013)

- Broke an incoming stream into small RDD batches (micro-batches)
- API was RDD-based — no DataFrame/SQL support natively
- Difficult to mix with batch logic
- Limited support for event-time processing and late data
- **Status: legacy, still works but not recommended**

```python
# DStream style (legacy)
from pyspark.streaming import StreamingContext
ssc = StreamingContext(sc, 1)           # 1-second batches
lines = ssc.socketTextStream("localhost", 9999)
counts = lines.flatMap(lambda l: l.split())\
              .map(lambda w: (w, 1))\
              .reduceByKey(lambda a,b: a+b)
counts.pprint()
```

### Generation 2 — Structured Streaming (Spark 2.0+, 2016)

- Stream treated as an **unbounded DataFrame** — same API as batch
- Full DataFrame / SQL / Dataset API
- Built-in event-time support and watermarking
- Exactly-once end-to-end semantics with checkpointing
- Easy mixing of streaming and static DataFrames
- **Status: current standard, what we use in this notebook**

```python
# Structured Streaming style (modern)
stream_df = spark.readStream.format("rate").load()
query = stream_df.writeStream.format("console").start()
```

| Feature | DStreams | Structured Streaming |
|---|---|---|
| API | RDD-based | DataFrame / SQL |
| Event-time | Manual, complex | Built-in |
| Late data | No native support | Watermarking |
| Exactly-once | Limited | Yes (with checkpointing) |
| Static join | Complex | Native |
| Continuous mode | No | Yes (experimental) |
| Status | Legacy | **Recommended** |

<h2 style="color:#E25822">3. Setup</h2>

Install PySpark and create a `SparkSession` with settings suitable for streaming demos.

In [ ]:
# Install PySpark (only needed on Google Colab / fresh environments)
!pip install pyspark --quiet

In [ ]:
import time
import os
import shutil
import random
import csv
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType
)

print("Imports successful")

In [ ]:
spark = (
    SparkSession.builder
    .appName("SparkStructuredStreaming-Demo")
    # Reduce shuffle partitions for small demo data
    .config("spark.sql.shuffle.partitions", "2")
    # Checkpoint location used by several examples below
    .config("spark.sql.streaming.checkpointLocation", "/tmp/spark_checkpoints")
    # Suppress noisy INFO logs
    .config("spark.driver.extraJavaOptions", "-Dlog4j.logger.org.apache.spark=WARN")
    .master("local[2]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"Python version: {spark.sparkContext.pythonVer}")
print("SparkSession ready.")

<h2 style="color:#E25822">Part 1 — Rate Source: Hello Streaming World</h2>

The **rate source** is the simplest built-in streaming source in Spark. It generates rows at a configurable rate — each row contains:

- `timestamp` — the event time of the row
- `value` — a monotonically increasing integer counter (0, 1, 2, …)

It is ideal for testing and learning because it needs no external infrastructure.

```
Rate Source generates:
┌───────────────────────────┬───────┐
│        timestamp          │ value │
├───────────────────────────┼───────┤
│ 2024-01-15 10:00:00.000   │   0   │
│ 2024-01-15 10:00:01.000   │   1   │
│ 2024-01-15 10:00:02.000   │   2   │
│          ...              │  ...  │
└───────────────────────────┴───────┘
```

### Output Modes

Structured Streaming supports three output modes that control which rows are written to the sink:

| Mode | Description | When to use |
|---|---|---|
| **append** | Only newly added rows (default) | No aggregation, or append-only sinks |
| **complete** | Entire result table every trigger | Aggregations where full state is needed |
| **update** | Only rows that changed since last trigger | Most aggregations; efficient |

### The `foreachBatch` Pattern

Instead of writing to an external sink (Kafka, database…), `foreachBatch` lets you write arbitrary Python/Spark code on each micro-batch. This is perfect for notebooks: we can call `.show()` on the batch DataFrame.

In [ ]:
# ── Part 1: Rate Source ────────────────────────────────────────────────────
#
# Read a rate stream: 2 rows per second
rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)   # generate 2 rows every second
    .option("numPartitions", 1)   # single partition for simple demo
    .load()
)

print("Schema of the rate source:")
rate_stream.printSchema()
print(f"Is streaming: {rate_stream.isStreaming}")

In [ ]:
# ── Aggregate: count rows received per second window ──────────────────────
#
# We truncate the timestamp to the nearest second, then count how many rows
# fell in that second. This is a simple tumbling-window aggregation.

batch_results = []   # We'll collect a few micro-batch results here

def process_rate_batch(batch_df, batch_id):
    """Called once per micro-batch. batch_df is a regular (static) DataFrame."""
    count = batch_df.count()
    if count == 0:
        return
    # Compute rows per second within this batch
    agg = (
        batch_df
        .withColumn("second", F.date_trunc("second", F.col("timestamp")))
        .groupBy("second")
        .agg(F.count("*").alias("rows_in_second"))
        .orderBy("second")
    )
    print(f"\n─── Micro-batch {batch_id} (total rows: {count}) ───")
    agg.show(truncate=False)
    batch_results.append(batch_id)

# Start the streaming query
query_rate = (
    rate_stream
    .writeStream
    .outputMode("append")          # append mode: pass raw rows to foreachBatch
    .foreachBatch(process_rate_batch)
    .trigger(processingTime="2 seconds")   # trigger every 2 seconds
    .option("checkpointLocation", "/tmp/chk_rate")
    .start()
)

print("Streaming query started. Collecting 3 micro-batches (~6 seconds)...")
time.sleep(8)          # let it run for ~8 seconds
query_rate.stop()      # always stop the query in notebook demos
print("\nQuery stopped. Total batches processed:", len(batch_results))

**What you should see above:**  
Each micro-batch prints a small table like:

```
─── Micro-batch 0 (total rows: 4) ───
+─────────────────────+──────────────+
| second              | rows_in_second|
+─────────────────────+──────────────+
| 2024-01-15 10:00:00 | 2            |
| 2024-01-15 10:00:01 | 2            |
+─────────────────────+──────────────+
```

Two rows per second — exactly what we configured. In a production system this aggregated data would be written to a database or dashboard.

<h2 style="color:#E25822">Part 2 — File-Based Streaming: Sensor Data</h2>

A common real-world pattern is **file-based streaming**: an external process drops new files into a directory, and Spark detects and processes them automatically. Spark remembers which files it has already processed, so it never reads the same file twice.

```
  External process               Spark Structured Streaming
  ──────────────                 ──────────────────────────
  sensor_001.csv ──▶ /tmp/       Detects new file ──▶ process
  sensor_002.csv ──▶ /tmp/       Detects new file ──▶ process
  sensor_003.csv ──▶ /tmp/       Detects new file ──▶ process
                                 (already seen files are skipped)
```

In this example we simulate the external process by writing CSV files ourselves.

**Scenario:** Factory floor sensors report temperature every few seconds. We want to:
1. Detect **anomalies** (temperature > 80°C)
2. Compute the **average temperature per device** per micro-batch

In [ ]:
# ── Setup directories ──────────────────────────────────────────────────────
INPUT_DIR  = "/tmp/streaming_input"
CHKPT_DIR  = "/tmp/chk_file_stream"

# Clean up any previous runs
for d in [INPUT_DIR, CHKPT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

print(f"Input directory  : {INPUT_DIR}")
print(f"Checkpoint dir   : {CHKPT_DIR}")

# ── Schema (must be declared for file streaming) ──────────────────────────
sensor_schema = StructType([
    StructField("device_id",   StringType(),    False),
    StructField("temperature", DoubleType(),    False),
    StructField("humidity",    DoubleType(),    True),
    StructField("event_time",  StringType(),    False),   # string; we'll cast later
])

print("\nSensor schema:")
for f in sensor_schema.fields:
    print(f"  {f.name:<15} {f.dataType}")

In [ ]:
# ── Data generator: writes one CSV file per call ──────────────────────────

DEVICES = [f"sensor_{i:03d}" for i in range(1, 6)]   # sensor_001 … sensor_005

def generate_sensor_file(file_index: int, num_rows: int = 10) -> str:
    """
    Writes a CSV file with simulated sensor readings.
    Occasionally injects an anomalous temperature (> 80°C).
    Returns the path of the written file.
    """
    path = os.path.join(INPUT_DIR, f"batch_{file_index:04d}.csv")
    base_time = datetime.utcnow() - timedelta(seconds=num_rows)

    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["device_id", "temperature", "humidity", "event_time"])
        for i in range(num_rows):
            device = random.choice(DEVICES)
            # Inject anomaly ~15% of the time
            if random.random() < 0.15:
                temp = round(random.uniform(81, 100), 2)
            else:
                temp = round(random.uniform(20, 79), 2)
            humidity  = round(random.uniform(30, 90), 2)
            ts = (base_time + timedelta(seconds=i)).strftime("%Y-%m-%d %H:%M:%S")
            writer.writerow([device, temp, humidity, ts])

    return path

# Write the first batch of files before starting the stream
for idx in range(3):
    p = generate_sensor_file(idx)
    print(f"Written: {p}")

print("\nInitial files ready.")

In [ ]:
# ── Read the streaming directory ───────────────────────────────────────────
sensor_stream = (
    spark.readStream
    .format("csv")
    .schema(sensor_schema)
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)   # process one file per micro-batch
    .load(INPUT_DIR)
)

# Cast event_time string to proper timestamp
sensor_stream = sensor_stream.withColumn(
    "event_time", F.to_timestamp(F.col("event_time"), "yyyy-MM-dd HH:mm:ss")
)

print("Streaming DataFrame schema:")
sensor_stream.printSchema()

In [ ]:
# ── Processing logic ──────────────────────────────────────────────────────
#
# For each micro-batch:
#   1. Print anomalies (temp > 80)
#   2. Print average temperature per device

def process_sensor_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    print(f"\n{'='*60}")
    print(f"  Micro-batch {batch_id}  |  Rows: {batch_df.count()}")
    print(f"{'='*60}")

    # ── 1. Anomaly Detection ──────────────────────────────────────
    anomalies = batch_df.filter(F.col("temperature") > 80)
    anomaly_count = anomalies.count()
    print(f"\n  [ALERT] {anomaly_count} anomalous reading(s) detected (temp > 80):")
    if anomaly_count > 0:
        anomalies.select("device_id", "temperature", "event_time").show(truncate=False)
    else:
        print("  No anomalies in this batch.")

    # ── 2. Average temperature per device ────────────────────────
    avg_temp = (
        batch_df
        .groupBy("device_id")
        .agg(
            F.round(F.avg("temperature"), 2).alias("avg_temp_C"),
            F.count("*").alias("readings")
        )
        .orderBy("device_id")
    )
    print("\n  Average temperature per device (this batch):")
    avg_temp.show(truncate=False)

# ── Start the streaming query ─────────────────────────────────────────────
query_file = (
    sensor_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(process_sensor_batch)
    .option("checkpointLocation", CHKPT_DIR)
    .start()
)

print("File-stream query started. Waiting for initial files to process...")
time.sleep(6)

# Simulate 2 more file arrivals while the query is running
for idx in range(3, 5):
    p = generate_sensor_file(idx)
    print(f"\n[SIM] New file arrived: {p}")
    time.sleep(3)

time.sleep(4)
query_file.stop()
print("\nQuery stopped.")

<h2 style="color:#E25822">Part 3 — Aggregations and Window Operations</h2>

Window functions let you group events that fall within a time range, rather than grouping by a static column. This is essential for questions like *"how many events happened in the last 10 seconds?"*

### Tumbling Windows

Non-overlapping, contiguous time buckets. Each event belongs to exactly **one** window.

```
TUMBLING WINDOW — size = 10 seconds

Events: e1  e2    e3  e4 e5        e6  e7
        |   |     |   |  |         |   |
────────┼───┼─────┼───┼──┼─────────┼───┼────────▶ time
  :00   :02 :04   :12 :14:16       :22 :24  :30

Window [00:00 – 00:10)  contains: e1, e2
Window [00:10 – 00:20)  contains: e3, e4, e5
Window [00:20 – 00:30)  contains: e6, e7
```

### Sliding Windows

Windows of fixed **size** that advance by a smaller **slide** interval. Events can belong to **multiple** windows.

```
SLIDING WINDOW — size = 20 seconds, slide = 10 seconds

Events: e1  e2    e3  e4 e5        e6  e7
        |   |     |   |  |         |   |
────────┼───┼─────┼───┼──┼─────────┼───┼────────▶ time
  :00   :02 :04   :12 :14:16       :22 :24  :30

Window [00:00 – 00:20)  contains: e1, e2, e3, e4, e5
Window [00:10 – 00:30)  contains: e3, e4, e5, e6, e7
Window [00:20 – 00:40)  contains: e6, e7

▲ e3, e4, e5 appear in TWO windows (overlap)
```

In Spark Structured Streaming you simply pass a **window spec** to `groupBy`:
- `F.window("event_time", "10 seconds")` — tumbling
- `F.window("event_time", "20 seconds", "10 seconds")` — sliding

In [ ]:
# ── Generate time-series data for window demo ──────────────────────────────
#
# Strategy: use spark.readStream.format("rate") and derive event_time from
# the built-in timestamp column. We assign a random event type to each row.

EVENT_TYPES = ["click", "view", "purchase", "search"]

# Rate stream: 5 rows per second
window_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    # Map the integer value to an event type using modulo
    .withColumn(
        "event_type",
        F.element_at(
            F.array([F.lit(e) for e in EVENT_TYPES]),
            (F.col("value") % len(EVENT_TYPES) + 1).cast(IntegerType())
        )
    )
    .withColumnRenamed("timestamp", "event_time")
)

print("Window stream schema:")
window_stream.printSchema()

In [ ]:
# ── Tumbling Window: count events per 10-second bucket ────────────────────

tumbling_agg = (
    window_stream
    .groupBy(
        F.window("event_time", "10 seconds"),   # tumbling window
        "event_type"
    )
    .agg(F.count("*").alias("event_count"))
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_type",
        "event_count"
    )
    .orderBy("window_start", "event_type")
)

def show_tumbling_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    print(f"\n─── Tumbling Window Batch {batch_id} ───")
    batch_df.show(20, truncate=False)

query_tumbling = (
    tumbling_agg
    .writeStream
    .outputMode("complete")            # complete mode required for aggregations
    .foreachBatch(show_tumbling_batch)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "/tmp/chk_tumbling")
    .start()
)

print("Tumbling window query running for ~15 seconds...")
time.sleep(18)
query_tumbling.stop()
print("Query stopped.")

In [ ]:
# ── Sliding Window: 20-second window, 5-second slide ──────────────────────
#
# Same data, but now each event can appear in up to 4 windows (20/5).
# We aggregate total events across all types to keep output compact.

sliding_agg = (
    window_stream
    .groupBy(
        F.window("event_time", "20 seconds", "5 seconds")   # sliding window
    )
    .agg(F.count("*").alias("total_events"))
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "total_events"
    )
    .orderBy("window_start")
)

def show_sliding_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    print(f"\n─── Sliding Window Batch {batch_id} ───")
    batch_df.show(20, truncate=False)

query_sliding = (
    sliding_agg
    .writeStream
    .outputMode("complete")
    .foreachBatch(show_sliding_batch)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "/tmp/chk_sliding")
    .start()
)

print("Sliding window query running for ~15 seconds...")
time.sleep(18)
query_sliding.stop()
print("Query stopped.")

**Interpreting the output:**

- **Tumbling**: Each `window_start`/`window_end` pair is unique and non-overlapping. Counts reflect exactly the events that arrived within that 10-second bucket.
- **Sliding**: The same timestamp range appears in multiple windows (overlap). Notice that counts grow larger as each window spans 20 seconds of data.

In `complete` output mode Spark re-emits the full aggregation table every trigger. As the stream runs longer, you will see counts accumulate.

<h2 style="color:#E25822">Part 4 — Watermarking: Handling Late Data</h2>

In a perfect world, events arrive in strict timestamp order. In reality, network delays and retries mean events arrive **late**.

```
LATE DATA PROBLEM

Event e3 has event_time = 10:00:05 but arrives at processing_time = 10:00:25

Processing time ────────────────────────────────────────────────▶
                 :00     :10     :20     :30

Arrived on time: e1(ts=:02)  e2(ts=:07)         e4(ts=:22)
Arrived LATE:                         e3(ts=:05) — 20 seconds late!

Window [00:00–00:10) already closed and emitted when e3 arrives.
Without watermarking: e3 is DROPPED or causes state to grow forever.
With watermarking   : e3 is accepted if it arrives within the allowed
                      lateness threshold.
```

### How Watermarking Works

A **watermark** is a moving threshold that tells Spark:
> *"I will accept events up to X minutes late. Events older than (max_event_time − X) can be discarded."*

```python
stream.withWatermark("event_time", "10 minutes")
```

Spark tracks the **maximum event_time seen so far** and sets the watermark to `max_event_time − 10 minutes`. Any event with `event_time < watermark` is dropped. Windows older than the watermark are finalized and their state is freed from memory — critical for long-running jobs.

```
  Max event_time seen = 10:00:30
  Watermark threshold = 10 minutes
  ─────────────────────────────────
  Watermark           = 10:00:30 − 10 min = 09:50:30

  Event with ts = 09:55:00 → ACCEPTED  (after watermark)
  Event with ts = 09:45:00 → DROPPED   (before watermark)
```

In [ ]:
# ── Watermarking demo ─────────────────────────────────────────────────────
#
# We use the rate source and deliberately inject a "late" event by
# subtracting time from the event_time column to simulate a delayed arrival.

# Rate stream as base
wm_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 3)
    .load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumn(
        "event_type",
        F.when(F.col("value") % 10 == 0, F.lit("LATE_EVENT"))
         .otherwise(F.lit("normal"))
    )
    # Simulate late arrival: for LATE_EVENT rows, backdate the event_time by 30s
    .withColumn(
        "event_time",
        F.when(
            F.col("event_type") == "LATE_EVENT",
            F.col("event_time") - F.expr("INTERVAL 30 SECONDS")
        ).otherwise(F.col("event_time"))
    )
)

# Apply watermark: accept events up to 15 seconds late
wm_agg = (
    wm_stream
    .withWatermark("event_time", "15 seconds")   # ← watermark declaration
    .groupBy(
        F.window("event_time", "10 seconds"),
        "event_type"
    )
    .agg(F.count("*").alias("count"))
    .select(
        F.col("window.start").alias("win_start"),
        F.col("window.end").alias("win_end"),
        "event_type",
        "count"
    )
    .orderBy("win_start", "event_type")
)

def show_wm_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    print(f"\n─── Watermark Batch {batch_id} ───")
    # Highlight LATE_EVENT rows
    batch_df.show(30, truncate=False)

query_wm = (
    wm_agg
    .writeStream
    .outputMode("update")              # update mode: emit only changed rows
    .foreachBatch(show_wm_batch)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "/tmp/chk_watermark")
    .start()
)

print("Watermarking query running for ~15 seconds...")
print("Look for LATE_EVENT rows — they have earlier window start times.")
time.sleep(18)
query_wm.stop()
print("\nQuery stopped.")

**Key observations:**

- `LATE_EVENT` rows appear in **earlier windows** than normal rows in the same batch — their event_time was backdated by 30 seconds.
- With `withWatermark("event_time", "15 seconds")` Spark holds window state open long enough to accept events up to 15 seconds late.
- Events backdated by 30 seconds exceed the watermark and are dropped — protecting memory in long-running jobs.
- In `update` output mode only windows that received new events in this batch are emitted, which is more efficient than `complete` mode for large aggregations.

> **Production tip:** Set the watermark to match your observed late-data distribution. Too tight → late events dropped. Too loose → excessive state memory. A common starting point is `2 × typical_network_delay`.

<h2 style="color:#E25822">Part 5 — Streaming Word Count (Classic Example)</h2>

Word count is the "Hello World" of distributed computing. Here we build a streaming version that maintains a **running total** of word occurrences using `complete` output mode.

**Idea:**
1. Rate source produces integer values
2. Map each integer to a word using modulo indexing over a vocabulary list
3. Group by word and count — in `complete` mode Spark shows the full running totals every trigger

```
Rate value → word mapping (vocabulary size = 8)
  0 → "spark"      1 → "data"       2 → "stream"    3 → "batch"
  4 → "window"     5 → "kafka"      6 → "hadoop"    7 → "pipeline"
  8 → "spark"  (wraps around via modulo)
```

In [ ]:
# ── Streaming Word Count ───────────────────────────────────────────────────

VOCABULARY = ["spark", "data", "stream", "batch",
              "window", "kafka", "hadoop", "pipeline"]

vocab_array = F.array([F.lit(w) for w in VOCABULARY])
vocab_size  = len(VOCABULARY)

word_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)     # 10 words per second
    .load()
    .withColumn(
        "word",
        F.element_at(vocab_array, (F.col("value") % vocab_size + 1).cast(IntegerType()))
    )
)

word_counts = (
    word_stream
    .groupBy("word")
    .agg(F.count("*").alias("count"))
    .orderBy(F.desc("count"))
)

batch_num = [0]   # mutable counter captured in closure

def show_word_counts(batch_df, batch_id):
    if batch_df.count() == 0:
        return
    batch_num[0] += 1
    print(f"\n─── Running Word Totals — Trigger {batch_num[0]} ───")
    batch_df.show(truncate=False)

query_wc = (
    word_counts
    .writeStream
    .outputMode("complete")           # complete: full table every trigger
    .foreachBatch(show_word_counts)
    .trigger(processingTime="4 seconds")
    .option("checkpointLocation", "/tmp/chk_wordcount")
    .start()
)

print("Word count query running for ~12 seconds...")
print("Counts should be roughly equal (uniform distribution).")
time.sleep(14)
query_wc.stop()
print("\nQuery stopped.")

**What to notice:**

- In `complete` mode each trigger prints the **entire** word count table, not just the deltas. Counts grow monotonically.
- Because we use modulo distribution all words appear at the same frequency — counts are approximately equal.
- In a real application words would come from a Kafka topic: `spark.readStream.format("kafka").option("subscribe", "chat-messages").load()`.

Compare this to `update` mode: only words whose count changed in the last batch would be emitted — more efficient for large vocabularies.

<h2 style="color:#E25822">Part 6 — Real-World Pattern: Streaming ETL</h2>

This section puts everything together in a pattern you will encounter in production:

```
STREAMING ETL PIPELINE

┌────────────────┐   filter    ┌─────────────────┐   join (static)  ┌───────────────────┐
│  Transaction   │ ──────────▶ │  Valid txns      │ ───────────────▶ │  Enriched txns    │
│  Stream        │  amt≤10000  │  (not fraud)     │  + category_name │  with full names  │
│                │             └─────────────────-┘                  └───────────────────┘
│                │   alert     ┌─────────────────┐                          │
│                │ ──────────▶ │  Fraud alerts   │                          │ window agg
│                │  amt>10000  │                 │                          ▼
└────────────────┘             └─────────────────┘                  ┌───────────────────┐
                                                                     │  Spend by category│
                                                                     │  per 30s window   │
                                                                     └───────────────────┘
```

### Steps:
1. **Generate** a transaction stream (user_id, amount, category_code, timestamp)
2. **Filter** fraudulent transactions (amount > 10 000)
3. **Enrich** by joining with a static lookup table (category_code → category_name)
4. **Aggregate** total spend by category in 30-second tumbling windows

### Production Sinks (not used in this demo, but common choices):

| Sink | Use case |
|---|---|
| **Kafka** | Forward enriched events to another service |
| **Delta Lake** | ACID-compliant data lake writes |
| **PostgreSQL / MySQL** | Operational database (via JDBC) |
| **Elasticsearch** | Full-text search and dashboards |
| **Console** | Debugging (what we use here) |

In [ ]:
# ── Static lookup table: category codes → human-readable names ────────────

category_data = [
    ("GROC", "Groceries"),
    ("ELEC", "Electronics"),
    ("TRVL", "Travel"),
    ("DINE", "Dining"),
    ("HLTH", "Health & Pharmacy"),
    ("ENTR", "Entertainment"),
]

category_lookup = spark.createDataFrame(
    category_data,
    schema=["category_code", "category_name"]
)

# Cache the static table so it's not re-computed for every micro-batch
category_lookup.cache()

print("Category lookup table:")
category_lookup.show()

In [ ]:
# ── Transaction stream via rate source ────────────────────────────────────
#
# We derive all transaction fields deterministically from the rate value
# so the stream is reproducible.

NUM_USERS  = 50
CATEGORIES = ["GROC", "ELEC", "TRVL", "DINE", "HLTH", "ENTR"]
cat_array  = F.array([F.lit(c) for c in CATEGORIES])

txn_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 8)
    .load()
    .withColumn(
        "user_id",
        F.concat(F.lit("user_"), (F.col("value") % NUM_USERS + 1).cast(StringType()))
    )
    .withColumn(
        "amount",
        # Mostly normal amounts; ~5% chance of a large fraudulent transaction
        F.when(
            F.col("value") % 20 == 0,
            (F.col("value") % 5000 + 10001).cast(DoubleType())
        ).otherwise(
            ((F.col("value") % 500 + 5) * 1.99).cast(DoubleType())
        )
    )
    .withColumn(
        "category_code",
        F.element_at(cat_array, (F.col("value") % len(CATEGORIES) + 1).cast(IntegerType()))
    )
    .withColumnRenamed("timestamp", "txn_time")
    .drop("value")
)

print("Transaction stream schema:")
txn_stream.printSchema()

In [ ]:
# ── foreachBatch: full ETL pipeline in each micro-batch ───────────────────

fraud_log   = []   # collect fraud events for post-run inspection
spend_log   = []   # collect spend summaries

def run_etl_batch(batch_df, batch_id):
    if batch_df.count() == 0:
        return

    print(f"\n{'━'*65}")
    print(f"  ETL Batch {batch_id}  |  Incoming rows: {batch_df.count()}")
    print(f"{'━'*65}")

    # ── Step 1: Fraud detection ────────────────────────────────────────
    fraud_txns = batch_df.filter(F.col("amount") > 10_000)
    normal_txns = batch_df.filter(F.col("amount") <= 10_000)

    fraud_count = fraud_txns.count()
    if fraud_count > 0:
        print(f"\n  ⚠  FRAUD ALERT: {fraud_count} suspicious transaction(s):")
        fraud_txns.select("user_id", "amount", "category_code", "txn_time").show(
            truncate=False
        )
        fraud_log.append(fraud_count)
    else:
        print(f"  No fraudulent transactions in this batch.")

    # ── Step 2: Enrich by joining with static lookup ───────────────────
    enriched = normal_txns.join(
        F.broadcast(category_lookup),   # broadcast the small lookup table
        on="category_code",
        how="left"
    )

    # ── Step 3: Aggregate — total spend by category ────────────────────
    spend_summary = (
        enriched
        .groupBy("category_name")
        .agg(
            F.round(F.sum("amount"), 2).alias("total_spend"),
            F.count("*").alias("txn_count"),
            F.round(F.avg("amount"), 2).alias("avg_spend")
        )
        .orderBy(F.desc("total_spend"))
    )

    print("\n  Spend by Category (this batch, fraud excluded):")
    spend_summary.show(truncate=False)
    spend_log.append(batch_id)

# ── Start the ETL streaming query ─────────────────────────────────────────
query_etl = (
    txn_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(run_etl_batch)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "/tmp/chk_etl")
    .start()
)

print("ETL streaming query running for ~20 seconds...")
time.sleep(22)
query_etl.stop()
print(f"\nQuery stopped. Batches processed: {len(spend_log)}")
print(f"Total fraud batches detected   : {len(fraud_log)}")
print(f"Total fraud transactions       : {sum(fraud_log)}")

**Pipeline recap:**

| Step | Operation | Spark API |
|---|---|---|
| Ingest | Read from rate source | `readStream.format("rate")` |
| Fraud filter | Split high-value transactions | `.filter(amount > 10000)` |
| Enrich | Join stream with static table | `.join(broadcast(lookup), ...)` |
| Aggregate | Sum spend by category | `.groupBy().agg(sum(...))` |
| Sink | Print to console | `foreachBatch` |

In production the `foreachBatch` body would instead:
- Write `fraud_txns` to a real-time alerting service or Kafka topic
- Write `enriched` to a Delta Lake table for downstream analytics
- Write `spend_summary` to a PostgreSQL table powering a live dashboard

The key insight: because `foreachBatch` gives you a static DataFrame, you can use **any** Spark or Python library inside it.

<h2 style="color:#E25822">Streaming vs Batch — When to Use What</h2>

| Question | If YES → | If NO → |
|---|---|---|
| Do you need results in < 1 minute? | **Streaming** | Batch may be fine |
| Is the dataset bounded (has a fixed end)? | **Batch** | Streaming |
| Do you need to process historical + live data together? | **Streaming** (with static join) | Batch |
| Is the processing logic simple aggregation? | Either works | Streaming gives lower latency |
| Do you need to detect patterns across multiple events over time? | **Streaming** with windows | Batch with full scan |
| Is your team new to distributed systems? | **Batch** first — simpler | Streaming when ready |
| Do events arrive continuously 24/7? | **Streaming** | Scheduled batch |
| Must you reprocess all historical data? | **Batch** | Streaming |

**Rule of thumb:** Start with batch. Move to streaming when latency requirements or continuous data arrival demand it. Avoid streaming complexity unless it provides clear business value.

<h2 style="color:#E25822">Key Concepts Summary</h2>

| Concept | Definition | Example |
|---|---|---|
| **Stream** | Unbounded, continuously arriving sequence of data | IoT sensor readings |
| **Batch** | Bounded, finite dataset processed at once | Yesterday's sales CSV |
| **Micro-batch** | Small batch of data collected over a short interval | Rows arriving in 1 second |
| **Trigger** | How often Spark processes new data | Every 5 seconds; once; continuous |
| **Source** | Where streaming data comes from | Kafka, rate, files, sockets |
| **Sink** | Where streaming results are written | Console, Delta Lake, Kafka, JDBC |
| **Output mode** | Which rows to write to the sink | append, complete, update |
| **Checkpoint** | Persistent state that enables fault tolerance and restart | `/tmp/spark_checkpoints` |
| **Watermark** | Threshold controlling how long Spark waits for late events | `withWatermark("ts", "10 min")` |
| **Window** | Time-based grouping of events | Tumbling 10s, sliding 20s/5s |
| **Event time** | Timestamp embedded in the event itself | `sensor_reading.timestamp` |
| **Processing time** | Wall-clock time when Spark processes the event | Spark system clock |
| **Late data** | Event arrives after its event-time window closed | Retry from network delay |
| **foreachBatch** | Hook to apply arbitrary code per micro-batch | Show, write to DB, alert |
| **State store** | In-memory store for ongoing aggregations | Counts, sums in complete mode |

<h2 style="color:#E25822">Exercises</h2>

Apply what you have learned by solving the following exercises. Each builds directly on the code in this notebook.

---

### Exercise 1 — Streaming Leaderboard

Using the rate source, build a streaming "game leaderboard":

1. Generate a stream at 5 rows/second.
2. Assign each row a `player_id` (map `value % 5` to names: `["Alice", "Bob", "Carol", "Dave", "Eve"]`).
3. Assign a `score` using `(value % 100) + 1` (1–100).
4. In each micro-batch compute: total score per player, number of rounds played, rank (using `F.rank()` window function ordered by total score descending).
5. Print the top-3 players per batch using `complete` output mode.
6. Run for 20 seconds.

**Expected output:** A table showing `player | total_score | rounds | rank` that updates every few seconds with growing totals.

---

### Exercise 2 — Anomaly Rate Monitor

Extend the file-based sensor streaming from Part 2:

1. Re-use the `sensor_schema` and `INPUT_DIR`.
2. Write a new batch processing function that calculates the **anomaly rate** per device: `anomaly_rate = anomalies / total_readings * 100`.
3. Flag any device where `anomaly_rate > 20%` with a `[HIGH RISK]` label.
4. Generate 6 new sensor files and stream-process them.
5. Show the per-device anomaly rate table sorted by `anomaly_rate` descending.

**Hint:** Use a single `groupBy("device_id")` with conditional aggregation:
```python
F.sum(F.when(F.col("temperature") > 80, 1).otherwise(0)).alias("anomaly_count")
```

---

### Exercise 3 — Late Data Investigation

Build a small experiment to understand watermark behaviour:

1. Create a rate stream at 4 rows/second.
2. Add a column `is_late` that is `True` for ~20% of rows (`value % 5 == 0`).
3. For late rows, backdate `event_time` by **60 seconds**.
4. Run the pipeline **twice** — once with `withWatermark("event_time", "30 seconds")` and once with `withWatermark("event_time", "90 seconds")`.
5. In each run, count how many `LATE` rows appear in the output.
6. Answer in a markdown cell: Which watermark threshold allowed more late events through? Why?

**Hint:** Late events have `event_time = current_time − 60s`. A 30-second watermark will drop them; a 90-second watermark will accept them.

In [ ]:
# ── Exercise 1 — Streaming Leaderboard ───────────────────────────────────
# Write your solution here.

from pyspark.sql.window import Window

PLAYERS = ["Alice", "Bob", "Carol", "Dave", "Eve"]
player_array = F.array([F.lit(p) for p in PLAYERS])

# TODO: build the stream, compute aggregations, rank players, writeStream
# Starter code:
leaderboard_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn(
        "player_id",
        F.element_at(player_array, (F.col("value") % len(PLAYERS) + 1).cast(IntegerType()))
    )
    .withColumn("score", (F.col("value") % 100 + 1).cast(IntegerType()))
)

# --- your aggregation + rank + writeStream logic goes here ---

print("Exercise 1 starter code ready. Complete the TODO above.")

In [ ]:
# ── Exercise 2 — Anomaly Rate Monitor ────────────────────────────────────
# Write your solution here.

# Hint: reuse INPUT_DIR and sensor_schema from Part 2.
# Generate new files with: generate_sensor_file(idx, num_rows=20)

def process_anomaly_rate_batch(batch_df, batch_id):
    """TODO: calculate anomaly_rate per device and flag HIGH RISK devices."""
    pass

print("Exercise 2 starter code ready. Implement process_anomaly_rate_batch above.")

In [ ]:
# ── Exercise 3 — Late Data Investigation ─────────────────────────────────
# Write your solution here.

# Hint: run the streaming query twice — change only the watermark duration.
# Track late event counts in a list and compare.

def run_watermark_experiment(watermark_duration: str, run_seconds: int = 15):
    """TODO: build stream with late data, apply watermark, count late rows accepted."""
    print(f"Running with watermark = {watermark_duration} for {run_seconds}s...")
    # --- your code here ---
    pass

# Uncomment to run both experiments:
# run_watermark_experiment("30 seconds")
# run_watermark_experiment("90 seconds")

print("Exercise 3 starter code ready. Implement run_watermark_experiment above.")

<h2 style="color:#E25822">Cleanup</h2>

Stop all active streaming queries and clean up temporary files.

In [ ]:
# ── Stop any remaining active queries ─────────────────────────────────────
active = spark.streams.active
if active:
    print(f"Stopping {len(active)} active query(ies)...")
    for q in active:
        q.stop()
        print(f"  Stopped: {q.name or q.id}")
else:
    print("No active queries to stop.")

# ── Remove temporary directories ──────────────────────────────────────────
tmp_dirs = [
    "/tmp/streaming_input",
    "/tmp/spark_checkpoints",
    "/tmp/chk_rate",
    "/tmp/chk_file_stream",
    "/tmp/chk_tumbling",
    "/tmp/chk_sliding",
    "/tmp/chk_watermark",
    "/tmp/chk_wordcount",
    "/tmp/chk_etl",
]

for d in tmp_dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed: {d}")

print("\nAll cleaned up. SparkSession still active for further exploration.")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

<h2 style="color:#E25822">Further Reading</h2>

| Resource | Description |
|---|---|
| [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html) | Official Spark documentation — comprehensive and up to date |
| [Delta Lake](https://delta.io/) | ACID-compliant storage layer; the most common sink for Spark streaming |
| [Apache Kafka](https://kafka.apache.org/) | Distributed message broker; most common streaming source in production |
| [Spark: The Definitive Guide](https://www.oreilly.com/library/view/spark-the-definitive/9781491912201/) | Chapters 20–23 cover Structured Streaming in depth |
| [Databricks Streaming Docs](https://docs.databricks.com/structured-streaming/index.html) | Managed Spark streaming with Delta Lake integration |

---

<div style="background:#f8f8f8; border-left:4px solid #E25822; padding:12px; border-radius:4px; margin-top:20px">
<strong>Module 5 — Spark Structured Streaming</strong><br>
Big Data Engineering Course | Topics covered: streaming vs batch, rate source, file streaming,
window operations, watermarking, streaming ETL.
</div>